# Fase 7 — Interpretabilidad comparada: Transformers vs. ML y entre variantes

Este notebook **no entrena nada**. Reúne los resultados de SHAP de:

- los notebooks 04 de `BETO`, `DistilBETO` y `RoBERTuito` (`<modelo>_shap_global.csv` y `<modelo>_shap_tokens.csv`);
- el notebook 02 de ML (`ml_shap_global.csv` y `ml_shap_rasgos.csv`: LR, RF y XGBoost).

Responde tres preguntas:

1. **¿Qué palabras empujan hacia la ironía en cada variante?** (tabla por modelo).
2. **¿Los modelos miran lo mismo?** Coincidencia (Jaccard) entre las 20 palabras más "irónicas" de cada modelo, por variante. Una coincidencia alta entre los Transformers ayuda a explicar por qué la votación no mejoró.
3. **¿Qué tiene de distinto Cuba?** Palabras compartidas por las tres variantes y propias de cada una (RoBERTuito y LR), y efecto de rasgos como risas, emojis, menciones o signos de puntuación.

**Advertencia de comparación:** los Transformers trabajan con sub-palabras de su tokenizador y el ML con palabras completas, raíces (`stem`) o lemas, según el preprocesamiento elegido. Para comparar se normaliza el texto (minúsculas, sin tildes y sin `#`/`@`), así que la coincidencia entre familias de modelos es **aproximada**; entre Transformers y entre variantes es más directa. Si falta algún archivo, ese modelo se omite. Resultados en `../data/resultados_interpretabilidad/`.

In [ ]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_DIR = Path("../data")
OUT_DIR = DATA_DIR / "resultados_interpretabilidad"
OUT_DIR.mkdir(parents=True, exist_ok=True)
VARIANTES = ["mx", "es", "cu"]
TRANSFORMERS = {"BETO": "beto", "DistilBETO": "distilbeto", "RoBERTuito": "robertuito"}
ML_DIR = DATA_DIR / "resultados_ml_groupcv_tfidf_lr_rf_10k"
TOP_N = 20  # palabras "más irónicas" por modelo para la coincidencia
MODELO_PRINCIPAL = "RoBERTuito"
ML_PRINCIPAL = "LR"


def normalizar(texto):
    t = unicodedata.normalize("NFKD", str(texto).lower().strip())
    t = "".join(ch for ch in t if not unicodedata.combining(ch))
    return t.lstrip("#@").strip()


GLOBAL, TOKENS = [], []
for nombre, pref in TRANSFORMERS.items():
    carpeta = DATA_DIR / f"resultados_{pref}_grouped_5x3_tpe20"
    g, t = carpeta / f"{pref}_shap_global.csv", carpeta / f"{pref}_shap_tokens.csv"
    if g.exists():
        d = pd.read_csv(g)
        d = d.rename(columns={"token_normalizado": "termino"}).assign(modelo=nombre, familia="Transformer")
        GLOBAL.append(d[["variante", "modelo", "familia", "termino", "n_textos", "shap_medio", "shap_abs_medio"]])
    else:
        print(f"AVISO: falta {g} (ejecuta el notebook 04 de {nombre}).")
    if t.exists():
        TOKENS.append(pd.read_csv(t, dtype={"ID": str}).assign(modelo=nombre))
g_ml = ML_DIR / "ml_shap_global.csv"
if g_ml.exists():
    d = pd.read_csv(g_ml).rename(columns={"palabra": "termino"}).assign(familia="ML")
    GLOBAL.append(d[["variante", "modelo", "familia", "termino", "n_textos", "shap_medio", "shap_abs_medio"]])
else:
    print(f"AVISO: falta {g_ml} (ejecuta el notebook 02, sección 11).")
if not GLOBAL:
    raise FileNotFoundError("No hay resultados de SHAP. Ejecuta primero los notebooks 02 y 04.")
GLOBAL = pd.concat(GLOBAL, ignore_index=True)
GLOBAL["termino_norm"] = GLOBAL["termino"].map(normalizar)
GLOBAL = GLOBAL[GLOBAL["termino_norm"].str.len() > 1]
TOKENS = pd.concat(TOKENS, ignore_index=True) if TOKENS else pd.DataFrame()
MODELOS = [m for m in [*TRANSFORMERS, "LR", "RF", "XGBoost"] if (GLOBAL["modelo"] == m).any()]
print("Modelos con SHAP:", MODELOS)

## 1. Palabras que más empujan hacia la ironía, por variante

Las `10` palabras con mayor contribución SHAP media hacia *irónico* (solo palabras presentes en suficientes tuits; ver cada notebook). Archivo: `top_palabras_ironia.csv`.

In [ ]:
filas = []
for variante in VARIANTES:
    for modelo in MODELOS:
        g = GLOBAL[(GLOBAL["variante"] == variante) & (GLOBAL["modelo"] == modelo)]
        top = g[g["shap_medio"] > 0].nlargest(10, "shap_medio")
        for rango, fila in enumerate(top.itertuples(), 1):
            filas.append({"variante": variante, "modelo": modelo, "puesto": rango, "termino": fila.termino,
                          "shap_medio": fila.shap_medio, "n_textos": fila.n_textos})
TOP = pd.DataFrame(filas)
TOP.to_csv(OUT_DIR / "top_palabras_ironia.csv", index=False)
for variante in VARIANTES:
    print(f"\n{variante.upper()}")
    display(TOP[TOP["variante"] == variante].pivot(index="puesto", columns="modelo", values="termino").reindex(columns=MODELOS))

## 2. ¿Los modelos miran las mismas palabras? (coincidencia de Jaccard)

Para cada variante se toman las `TOP_N` palabras con mayor contribución hacia *irónico* de cada modelo y se calcula la coincidencia de Jaccard (|A ∩ B| / |A ∪ B|; 1 = mismas palabras, 0 = ninguna en común). Archivos: `coincidencia_jaccard.csv` y `coincidencia_jaccard.png/.pdf`.

In [ ]:
def conjunto(variante, modelo):
    g = GLOBAL[(GLOBAL["variante"] == variante) & (GLOBAL["modelo"] == modelo) & (GLOBAL["shap_medio"] > 0)]
    return set(g.nlargest(TOP_N, "shap_medio")["termino_norm"])


filas = []
fig, axes = plt.subplots(1, len(VARIANTES), figsize=(5.0 * len(VARIANTES), 4.6), squeeze=False)
for ax, variante in zip(axes[0], VARIANTES):
    M = np.full((len(MODELOS), len(MODELOS)), np.nan)
    for a, ma in enumerate(MODELOS):
        for b, mb in enumerate(MODELOS):
            A, B = conjunto(variante, ma), conjunto(variante, mb)
            if A and B:
                M[a, b] = len(A & B) / len(A | B)
                if a < b:
                    filas.append({"variante": variante, "modelo_a": ma, "modelo_b": mb, "jaccard": M[a, b],
                                  "comunes": ", ".join(sorted(A & B))})
    im = ax.imshow(M, cmap="Greens", vmin=0, vmax=1)
    for a in range(len(MODELOS)):
        for b in range(len(MODELOS)):
            if not np.isnan(M[a, b]):
                ax.text(b, a, f"{M[a, b]:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if M[a, b] > 0.6 else "black")
    ax.set_xticks(range(len(MODELOS)), MODELOS, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(MODELOS)), MODELOS, fontsize=8)
    ax.set_title(variante.upper())
fig.colorbar(im, ax=axes[0].tolist(), shrink=0.8, label=f"Jaccard (top {TOP_N} hacia irónico)")
fig.suptitle("Coincidencia entre modelos en las palabras que empujan hacia la ironía", fontsize=12)
fig.savefig(OUT_DIR / "coincidencia_jaccard.png", dpi=200, bbox_inches="tight")
fig.savefig(OUT_DIR / "coincidencia_jaccard.pdf", bbox_inches="tight")
display(fig)
plt.close(fig)
JACCARD = pd.DataFrame(filas)
JACCARD.to_csv(OUT_DIR / "coincidencia_jaccard.csv", index=False)
display(JACCARD.drop(columns="comunes").round(3))

## 3. ¿Qué tiene de distinto cada variante?

Para RoBERTuito (modelo principal) y LR (ML más interpretable) se comparan las `TOP_N` palabras más "irónicas" de MX, ES y CU: cuáles comparten las tres variantes y cuáles aparecen solo en una. Archivo: `palabras_por_variante.csv`.

In [ ]:
filas = []
for modelo in [MODELO_PRINCIPAL, ML_PRINCIPAL]:
    if modelo not in MODELOS:
        continue
    conj = {v: conjunto(v, modelo) for v in VARIANTES}
    comunes = set.intersection(*conj.values()) if all(conj.values()) else set()
    print(f"\n{modelo} — compartidas por MX, ES y CU ({len(comunes)}): {', '.join(sorted(comunes)) or '—'}")
    for v in VARIANTES:
        propias = conj[v] - set().union(*(conj[o] for o in VARIANTES if o != v))
        print(f"{modelo} — solo en {v.upper()} ({len(propias)}): {', '.join(sorted(propias)) or '—'}")
        for termino in conj[v]:
            filas.append({"modelo": modelo, "variante": v, "termino": termino,
                          "compartida_por_las_3": termino in comunes, "solo_en_esta_variante": termino in propias})
pd.DataFrame(filas).to_csv(OUT_DIR / "palabras_por_variante.csv", index=False)

## 4. Rasgos del tuit: risas, emojis, menciones, hashtags y puntuación

Los modelos de ML usan 9 rasgos lingüísticos explícitos (`ml_shap_rasgos.csv`). Para los Transformers se agrupan sus tokens en categorías equivalentes con expresiones regulares (por ejemplo, `jaja…` → risas; los emojis aparecen como texto tras el preprocesamiento de `pysentimiento`) y se promedia su contribución SHAP **en los tuits donde aparecen**. Un valor positivo indica que el rasgo empuja hacia *irónico*. Archivo: `rasgos_por_modelo.csv`.

In [ ]:
CATEGORIAS = {
    "risas": r"^(?:ja|je|ji|jo|ha|he|xd|lol|jsj)+[aeijos]*$",
    "emojis": r"emoji",
    "menciones": r"^@?(?:usuario|user)$",
    "hashtags": r"^#|^hashtag$",
    "exclamaciones": r"^[!¡]+$",
    "interrogaciones": r"^[?¿]+$",
    "puntos suspensivos": r"^\.{2,}$|^…$",
    "comillas": r"^[\"“”«»']+$",
    "negaciones": r"^(?:no|nunca|jamas|jamás|tampoco|ni)$",
}
filas = []
if not TOKENS.empty:
    tok = TOKENS.copy()
    tok["t"] = tok["token"].astype(str).str.replace("Ġ", " ").str.replace("##", "").str.strip().str.lower()
    for categoria, patron in CATEGORIAS.items():
        m = tok[tok["t"].str.contains(patron, regex=True, na=False)]
        for (variante, modelo), g in m.groupby(["variante", "modelo"]):
            por_tuit = g.groupby("ID")["shap"].sum()  # contribución total del rasgo en cada tuit
            filas.append({"variante": variante, "modelo": modelo, "rasgo": categoria,
                          "n_textos": int(por_tuit.size), "shap_medio": float(por_tuit.mean())})
ruta_rasgos_ml = ML_DIR / "ml_shap_rasgos.csv"
if ruta_rasgos_ml.exists():
    r = pd.read_csv(ruta_rasgos_ml)
    filas += r[[c for c in ["variante", "modelo", "rasgo", "n_textos", "shap_medio"] if c in r.columns]].to_dict("records")
RASGOS = pd.DataFrame(filas)
RASGOS.to_csv(OUT_DIR / "rasgos_por_modelo.csv", index=False)
if not RASGOS.empty:
    tabla = RASGOS.pivot_table(index="rasgo", columns=["modelo", "variante"], values="shap_medio")
    signo = tabla.apply(lambda col: col.map(
        lambda x: "" if pd.isna(x) else ("+ irónico" if x > 0 else ("− no irónico" if x < 0 else "0"))))
    print("Dirección del efecto de cada rasgo (modelo × variante):")
    display(signo)
    display(tabla.round(4))

## 5. Cómo usar estos resultados en el artículo

- **Figuras principales:** `robertuito_shap_global.png` (palabras por variante) y `ml_shap_palabras.png` (LR, RF y XGBoost); en el anexo, las de BETO y DistilBETO y `coincidencia_jaccard.png`.
- **Ejemplos locales:** `robertuito_shap_ejemplos.png` y los HTML de `shap_html/` (un acierto, un falso positivo y un falso negativo por variante), junto con `errores_consistentes_robertuito.csv` del notebook 06.
- **Redacción:** describir qué palabras y rasgos empujan hacia la ironía en cada variante, qué comparte Cuba con MX y ES y qué no, y si los modelos coinciden (Jaccard). SHAP explica el comportamiento del modelo; no prueba causalidad lingüística ni cambia los resultados.